# 07 — Leakage-free rebuild of the predictor matrix, sensitivity analysis and the hold-out table

**Purpose.** Re-run *your own* `features.ipynb` pipeline with only the imputation and the fitted
statistics changed, so the feature construction stays identical:

| | submitted pipeline | revised pipeline |
|---|---|---|
| gaps in inputs | linear interpolation, `limit_direction='both'`, whole record | forward fill; leading gaps take the pre-2020 median |
| medians for infinite ratios, high/low-wind quantiles | whole record | days up to `STATS_END` only |
| missing targets | interpolated, then used | excluded from training and scoring |

Then reproduce the submitted 2024 hold-out table with the leakage-free matrix, and quantify the
difference (variants: published, rebuilt, forward-fill, + missingness indicators, complete cases).

**Answers:** Reviewer 3 comment 2; Reviewer 4 comment 4 (and part of 1).
**Outputs:** `revision/df_final_features_causal.feather`, `revision/original_protocol_predictions.csv`,
`revision/tables/S6_imputation_sensitivity.csv`, `T1_original_protocol_metrics.csv`, models in
`revision/models/`. **Runtime:** ~20–40 min (two full feature builds plus twelve model fits).

In [6]:
# --- Setup: Drive, paths, shared helpers -------------------------------------------------------
import os, sys, json, time, warnings, subprocess
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'shap', 'scienceplots',
                    'ephem', 'catboost', 'lightgbm'], check=False)
except Exception:
    pass
_BASE = os.environ.get('SARKOY_BASE', '/content/drive/MyDrive/windforecast_rev1/')
sys.path.insert(0, os.path.join(_BASE, 'revision'))
if not os.path.exists(os.path.join(_BASE, 'revision', 'revision_utils.py')):
    raise FileNotFoundError('Copy revision_utils.py into ' + os.path.join(_BASE, 'revision') + ' first.')
import revision_utils as ru
import numpy as np, pandas as pd, matplotlib.pyplot as plt
BASE, REV, IN_COLAB, FAST = ru.get_paths()
ru.set_style()
RES = ru.Results(REV)
RES.path = REV + 'results_07.json'
GPU = ru.has_gpu()
try:
    from IPython.display import display
except Exception:
    display = print
print('BASE:', BASE, '| REV:', REV, '| GPU:', GPU, '| FAST (smoke test):', FAST)
print(ru.log_versions(REV))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/windforecast_rev1/ | REV: /content/drive/MyDrive/windforecast_rev1/revision/ | GPU: True | FAST (smoke test): False
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'catboost': '1.2.10', 'optuna': '5.0.0', 'shap': '0.52.0', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0', 'seaborn': '0.13.2', 'pyarrow': '23.0.1', 'ephem': '4.2.1', 'tensorflow': '2.20.0', 'gpu': True}


In [7]:
# --- Configuration ------------------------------------------------------------------------------
STATS_END = '2019-12-31'      # statistics use only days before the first outer test year (2020)
TRAIN_END_YEAR, VAL_YEAR, TEST_YEAR = 2022, 2023, 2024
RUN_LSTM = True               # deep-learning baseline asked for by Reviewer 3
RUN_REDUCED = os.environ.get('SARKOY_SKIP_REDUCED', '0') != '1'   # two-stage reduced set (slow on one CPU)
GITHUB_FEATURES = 'https://raw.githubusercontent.com/intelMT/sarkoy-wind-forecasting/main/features.ipynb'
PUBLISHED_WINDOWS = [3, 7, 14, 21, 30]   # rolling windows behind the submitted model (best_xgboost_model.json)
print('statistics cut-off:', STATS_END, '| LSTM baseline:', RUN_LSTM)


statistics cut-off: 2019-12-31 | LSTM baseline: True


In [8]:
# --- Load the original feature-engineering notebook and patch two things only -------------------
import nbformat, re, io, shutil, contextlib, urllib.request
fn = next((f for f in [BASE + 'features.ipynb', REV + 'features.ipynb'] if os.path.exists(f)), None)
if fn is None:
    fn = REV + 'features.ipynb'
    urllib.request.urlretrieve(GITHUB_FEATURES, fn)
nbf = nbformat.read(fn, as_version=4)
code = [c.source for c in nbf.cells if c.cell_type == 'code']
last = max(i for i, s in enumerate(code) if 'df_final_features' in s and 'to_feather' in s)
pipeline = code[:last + 1]
PAT = re.compile(r"\.interpolate\(\s*method\s*=\s*'linear'\s*,\s*limit_direction\s*=\s*'both'\s*\)")
def transform(src, vdir):
    keep = []
    for ln in src.split('\n'):
        s = ln.strip()
        if s.startswith(('!', '%')) or 'drive.mount' in s or 'google.colab' in s:
            keep.append('# [revision run] ' + ln)
        else:
            keep.append(ln)
    src = '\n'.join(keep)
    src = re.sub(r"/content/drive/MyDrive/windforecast(?![A-Za-z0-9_])/?", lambda m: vdir, src)
    src = PAT.sub('.pipe(_REV_IMPUTE)', src)
    # the submitted model was trained on 3-, 7-, 14-, 21- and 30-day windows; the saved notebook lists
    # [5, 14, 21, 30], so the published window set is restored here (see the fidelity check below)
    src = src.replace('for window in [5, 14, 21, 30]:', 'for window in ' + str(PUBLISHED_WINDOWS) + ':')
    src = re.sub(r"df\[numeric_cols\]\.median\(\)", "_REV_STAT(df, numeric_cols, 'median')", src)
    src = re.sub(r"df\[(col|source_target_col)\]\.median\(\)", r"_REV_STAT(df, \1, 'median')", src)
    src = re.sub(r"df\[source_target_col\]\.quantile\(([0-9.]+)\)",
                 r"_REV_STAT(df, source_target_col, 'quantile', \1)", src)
    return src
print(f'{fn}: {len(pipeline)} pipeline cells of {len(code)}; '
      f'{sum(len(PAT.findall(s)) for s in pipeline)} interpolation calls patched')


/content/drive/MyDrive/windforecast_rev1/features.ipynb: 24 pipeline cells of 36; 3 interpolation calls patched


In [9]:
# --- The two imputation regimes -----------------------------------------------------------------
def _stat_factory(stats_end):
    def _stat(df, cols, how, q=None):
        sub = df
        if stats_end is not None and isinstance(df.index, pd.DatetimeIndex):
            s2 = df.loc[df.index <= pd.Timestamp(stats_end)]
            sub = s2 if len(s2) else df
        d = sub[cols]
        return d.median() if how == 'median' else d.quantile(q)
    return _stat

def _impute_original(o):
    if isinstance(o, pd.DataFrame):
        o = o.copy(); n = o.select_dtypes('number').columns
        o[n] = o[n].interpolate(method='linear', limit_direction='both')
        return o
    return o.interpolate(method='linear', limit_direction='both')

def _impute_causal_factory(stats_end):
    def _imp(o):
        if isinstance(o, pd.DataFrame):
            o = o.copy(); n = o.select_dtypes('number').columns
            med = o.loc[o.index <= pd.Timestamp(stats_end), n].median()
            o[n] = o[n].ffill().fillna(med)
            return o
        med = o.loc[o.index <= pd.Timestamp(stats_end)].median()
        return o.ffill().fillna(med)
    return _imp

VARIANTS = {'original': dict(_REV_IMPUTE=_impute_original, _REV_STAT=_stat_factory(None)),
            'causal': dict(_REV_IMPUTE=_impute_causal_factory(STATS_END), _REV_STAT=_stat_factory(STATS_END))}


In [10]:
# --- Run the pipeline once per variant ----------------------------------------------------------
def run_variant(name):
    vdir = os.path.join(REV, 'variants', name) + '/'
    os.makedirs(vdir, exist_ok=True)
    link = os.path.join(vdir, 'data')
    if not os.path.exists(link):
        try:
            os.symlink(os.path.join(BASE, 'data'), link)
        except OSError:
            shutil.copytree(os.path.join(BASE, 'data'), link)
    ns = {'__name__': '__main__'}
    ns.update(VARIANTS[name])
    log, t0 = io.StringIO(), time.time()
    try:
        with contextlib.redirect_stdout(log):
            for k, src in enumerate(pipeline):
                exec(compile(transform(src, vdir), f'<features cell {k}>', 'exec'), ns)
    finally:
        open(os.path.join(REV, 'logs', f'features_{name}.log'), 'w').write(log.getvalue())
    print(f'{name}: {time.time() - t0:.0f} s -> {vdir}df_final_features.feather')
    return vdir + 'df_final_features.feather'

paths = {v: run_variant(v) for v in VARIANTS}
PUBLISHED_AVAILABLE = os.path.exists(BASE + 'df_final_features.feather')
if not PUBLISHED_AVAILABLE:
    # no copy of the submitted matrix in this folder: use the re-executed original pipeline as the reference,
    # which makes the fidelity check below trivially zero (it is only meaningful on the original Drive folder)
    shutil.copy(paths['original'], BASE + 'df_final_features.feather')
    print('NOTE: submitted df_final_features.feather not found; the rebuilt original-pipeline matrix is used instead')


original: 13 s -> /content/drive/MyDrive/windforecast_rev1/revision/variants/original/df_final_features.feather
causal: 11 s -> /content/drive/MyDrive/windforecast_rev1/revision/variants/causal/df_final_features.feather


In [11]:
# --- Does the rebuild reproduce the submitted matrix? (fidelity check) --------------------------
def read_matrix(p):
    d = pd.read_feather(p)
    d = d.set_index(pd.to_datetime(d.pop('date_time'))).sort_index()
    d.index.name = 'date_time'
    return d
pub = ru.load_matrix(BASE, REV, 'published', verbose=False)
reb = read_matrix(paths['original'])
common_idx = pub.index.intersection(reb.index)
cols = [c for c in pub.columns if c in reb.columns and pd.api.types.is_numeric_dtype(pub[c])]
maxdiff = float((reb.loc[common_idx, cols] - pub.loc[common_idx, cols]).abs().to_numpy().max())
for nm, d in (('rebuilt', reb), ('submitted', pub)):
    miss = sorted(set(ru.FEATURES_559) - set(d.columns))
    print(f'{nm}: {len(miss)} of the 559 model predictors absent', miss[:6])
print('rebuilt shape', reb.shape, '| submitted shape', pub.shape, '| shared dates', len(common_idx),
      '| largest absolute difference', f'{maxdiff:.2e}')
RES.put('REBUILD_MAXDIFF', f'{maxdiff:.1e}' if PUBLISHED_AVAILABLE else 'n/a (submitted matrix not in this folder)')
RES.put('REBUILD_SAME_ROWS', 'yes' if reb.index.equals(pub.index) else 'no')


rebuilt: 0 of the 559 model predictors absent []
submitted: 0 of the 559 model predictors absent []
rebuilt shape (2964, 563) | submitted shape (2964, 564) | shared dates 2964 | largest absolute difference 2.27e-13


In [12]:
# --- Assemble the leakage-free matrix: observed-target flag and missingness indicators ----------
cau = read_matrix(paths['causal'])
numc = cau.select_dtypes('number').columns
bad = np.isinf(cau[numc].to_numpy()) | cau[numc].isna().to_numpy()
n_bad = int(bad.sum())
if n_bad:
    # a forward-filled gap makes a station series constant, so a 7-day correlation with it is undefined;
    # such values are set to 0 (no measurable association) and any other non-finite value is carried forward
    cau[numc] = cau[numc].replace([np.inf, -np.inf], np.nan)
    corr_cols = [c for c in numc if c.startswith('corr_')]
    cau[corr_cols] = cau[corr_cols].fillna(0.0)
    other = [c for c in numc if c not in corr_cols]
    cau[other] = cau[other].ffill().fillna(cau.loc[cau.index <= pd.Timestamp(STATS_END), other].median())
print('non-finite cells repaired in the leakage-free matrix:', n_bad)
for f in ru.FEATURES_559:
    if f not in cau.columns:
        if '_cat_' in f:
            cau[f] = 0.0      # a rare one-hot category that never occurs under forward filling
        else:
            raise KeyError('predictor missing from the rebuilt matrix: ' + f)
RES.put('CAUSAL_NONFINITE_CELLS', n_bad, 'd')
sar = ru.load_raw_sarkoy(BASE)
cau['target_observed'] = ru.observed_target_mask(cau.index, sar).to_numpy()
flags = (pd.read_feather(REV + 'missing_flags.feather').set_index('date_time')
         if os.path.exists(REV + 'missing_flags.feather') else ru.missing_flags(BASE))
flags = flags.reindex(cau.index).fillna(0).astype(int)
cau.reset_index().to_feather(REV + 'df_final_features_causal.feather')
print('leakage-free matrix:', cau.shape, '| target observed on', int(cau.target_observed.sum()), 'days',
      '| days with at least one imputed input:', int((flags.sum(axis=1) > 0).sum()))
RES.put('CAUSAL_N_ROWS', f'{len(cau):,}').put('CAUSAL_N_TARGET_DROPPED', int((~cau.target_observed).sum()), 'd')


non-finite cells repaired in the leakage-free matrix: 367
leakage-free matrix: (2964, 564) | target observed on 2957 days | days with at least one imputed input: 797


In [13]:
# --- Sensitivity of the 2024 result to the imputation regime ------------------------------------
def fit_eval(dfm, feats, label, keep=None, all_days=False):
    X = ru.get_X(dfm, feats); y = dfm[ru.TARGET]
    ok = pd.Series(True, index=dfm.index) if all_days else dfm['target_observed'].astype(bool)
    if keep is not None:
        ok = ok & keep.reindex(dfm.index).fillna(False).astype(bool)
    tr = ok & (dfm.index.year <= TRAIN_END_YEAR); va = ok & (dfm.index.year == VAL_YEAR)
    te = ok & (dfm.index.year == TEST_YEAR)
    m = ru.make_model('XGBoost_400', ru.PUBLISHED_PARAMS['XGBoost_400'], gpu=GPU, fast=FAST).fit(X[tr], y[tr])
    row = dict(variant=label, predictors=X.shape[1], n_train=int(tr.sum()), n_test=int(te.sum()))
    for nm, msk in (('val', va), ('test', te)):
        p = m.predict(X[msk])
        row[f'{nm}_RMSE'] = ru.rmse(y[msk], p); row[f'{nm}_MAE'] = ru.mae(y[msk], p)
        row[f'{nm}_R2'] = ru.r2(y[msk], p)
        row[f'{nm}_persistence_RMSE'] = ru.rmse(y[msk], dfm.loc[msk, ru.SOURCE])
    return row, m

pub2 = pub.copy(); pub2['target_observed'] = cau['target_observed'].reindex(pub2.index).fillna(False)
cau_ind = cau.join(flags)
rows = []
rows.append(fit_eval(pub2, ru.FEATURES_559, 'A. submitted pipeline, all days (as published)', all_days=True)[0])
rows.append(fit_eval(pub2, ru.FEATURES_559, 'B. submitted pipeline, observed targets only')[0])
rows.append(fit_eval(cau, ru.FEATURES_559, 'C. forward-fill pipeline (revised)')[0])
rows.append(fit_eval(cau_ind, ru.FEATURES_559 + list(flags.columns), 'D. revised + missingness indicators')[0])
rows.append(fit_eval(cau, ru.FEATURES_559, 'E. revised, complete cases only',
                     keep=(flags.sum(axis=1) == 0))[0])
sens = pd.DataFrame(rows)
sens.to_csv(REV + 'tables/S6_imputation_sensitivity.csv', index=False)
display(sens.round(4))
base_rmse = sens.test_RMSE.iloc[0]
sens['test_skill'] = 100 * (1 - sens.test_RMSE / sens.test_persistence_RMSE)
same_days = sens.iloc[:4]                      # A-D are scored on (almost) the same test days; E is not
RES.put('SENS_A_RMSE', sens.test_RMSE.iloc[0], '.4f').put('SENS_C_RMSE', sens.test_RMSE.iloc[2], '.4f')
RES.put('SENS_XGB_DELTA', sens.test_RMSE.iloc[2] - base_rmse, '+.4f')
RES.put('SENS_MAXABS_DELTA', float(np.max(np.abs(same_days.test_RMSE - base_rmse))), '.4f')
RES.put('SENS_A_SKILL', sens.test_skill.iloc[0], '.1f').put('SENS_C_SKILL', sens.test_skill.iloc[2], '.1f')
RES.put('SENS_E_NTEST', int(sens.n_test.iloc[4]), 'd').put('SENS_E_SKILL', sens.test_skill.iloc[4], '.1f')


,variant,predictors,n_train,n_test,val_RMSE,val_MAE,val_R2,val_persistence_RMSE,test_RMSE,test_MAE,test_R2,test_persistence_RMSE
0,"A. submitted pipeline, all days (as published)",559,2234,365,1.0009,0.7785,0.4958,1.3579,0.9537,0.7338,0.4675,1.1278
1,"B. submitted pipeline, observed targets only",559,2229,364,1.0060,0.7771,0.4920,1.3595,0.9572,0.7418,0.4632,1.1294
2,C. forward-fill pipeline (revised),559,2229,364,1.0145,0.7839,0.4833,1.3604,0.9562,0.7429,0.4644,1.1294
3,D. revised + missingness indicators,577,2229,364,1.0182,0.7855,0.4796,1.3604,0.9587,0.7476,0.4615,1.1294
4,"E. revised, complete cases only",559,1835,91,1.0336,0.7806,0.5373,1.3903,1.0614,0.8144,0.5043,1.1974


In [14]:
# --- The 2024 hold-out table, recomputed on the leakage-free matrix -----------------------------
X = ru.get_X(cau); y = cau[ru.TARGET]; ok = cau['target_observed'].astype(bool)
tr = ok & (cau.index.year <= TRAIN_END_YEAR); va = ok & (cau.index.year == VAL_YEAR); te = ok & (cau.index.year == TEST_YEAR)
base = ru.baseline_predictions(cau, tr)
preds = {'Persistence': base['persistence'], 'Seasonal-naive': base['seasonal_naive'],
         '7-day moving average': base['moving_average_7d']}
SPEC = [('LassoCV', 'LassoCV', None), ('RidgeCV', 'RidgeCV', None), ('ElasticNetCV', 'ElasticNetCV', None),
        ('SVR', 'SVR', ru.PUBLISHED_PARAMS['SVR']),
        ('Random Forest', 'RandomForest', ru.PUBLISHED_PARAMS['RandomForest']),
        ('LightGBM', 'LightGBM', ru.PUBLISHED_PARAMS['LightGBM']),
        ('CatBoost', 'CatBoost', ru.PUBLISHED_PARAMS['CatBoost']),
        ('XGBoost (50 trials)', 'XGBoost_50', ru.PUBLISHED_PARAMS['XGBoost_50']),
        ('XGBoost (400 trials)', 'XGBoost_400', ru.PUBLISHED_PARAMS['XGBoost_400'])]
fitted = {}
for label, name, params in SPEC:
    t0 = time.time()
    m = ru.make_model(name, params, gpu=GPU, fast=FAST).fit(X[tr], y[tr])
    preds[label] = pd.Series(m.predict(X), index=X.index)
    fitted[label] = m
    print(f'{label:24s} {time.time() - t0:6.0f} s  test RMSE {ru.rmse(y[te], preds[label][te]):.4f}')
if RUN_REDUCED:
    red = ru.reduce_features(X[tr], y[tr], fast=FAST)
    mr = ru.make_model('XGBoost_50', ru.PUBLISHED_PARAMS['XGBoost_50'], gpu=GPU, fast=FAST).fit(X.loc[tr, red], y[tr])
    preds['XGBoost (reduced set)'] = pd.Series(mr.predict(X[red]), index=X.index)
    print('reduced predictor set:', len(red), 'features ->', red[:6], '...')
    RES.put('N_REDUCED_SET', len(red), 'd')
os.makedirs(REV + 'models', exist_ok=True)
fitted['XGBoost (400 trials)'].save_model(REV + 'models/xgb400_causal.json')
import joblib; joblib.dump(fitted['Random Forest'], REV + 'models/rf_causal.joblib')


LassoCV                       6 s  test RMSE 1.1077
RidgeCV                      26 s  test RMSE 1.1472
ElasticNetCV                 68 s  test RMSE 1.0760
SVR                           3 s  test RMSE 1.0675
Random Forest               125 s  test RMSE 0.9741
LightGBM                      6 s  test RMSE 0.9659
CatBoost                     10 s  test RMSE 0.9706
XGBoost (50 trials)           4 s  test RMSE 0.9601
XGBoost (400 trials)         15 s  test RMSE 0.9562
reduced predictor set: 10 features -> ['max_speed_enez_vertical', 'max_speed_marmara_horizontal', 'abs_max_speed_marmara_horizontal', 'abs_max_speed_enez_vertical', 'mean_speed_enez_vertical', 'max_speed_lalapasa_vertical'] ...


['/content/drive/MyDrive/windforecast_rev1/revision/models/rf_causal.joblib']

In [15]:
# --- LSTM baseline (Reviewer 3, deep-learning comment) ------------------------------------------
if RUN_LSTM:
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, models, callbacks, optimizers
        tf.keras.utils.set_random_seed(42)
        sc = ru.MonthScaled(None)
        Xtr = sc._tx(X[tr], fit=True)[:, None, :]; Xva = sc._tx(X[va])[:, None, :]; Xall = sc._tx(X)[:, None, :]
        units, layers_n, drop = (16, 2, 0.1) if FAST else (416, 3, 0.09876585985584417)
        inp = layers.Input(shape=(1, X.shape[1])); h = inp
        for i in range(layers_n):
            h = layers.LSTM(units, return_sequences=(i < layers_n - 1))(h)
            h = layers.Dropout(drop)(h)
        net = models.Model(inp, layers.Dense(1)(h))
        net.compile(optimizer=optimizers.Adam(0.0012561075381468112), loss='mse')
        net.fit(Xtr, y[tr].to_numpy(), validation_data=(Xva, y[va].to_numpy()), epochs=2 if FAST else 100,
                batch_size=128, verbose=0,
                callbacks=[callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)])
        preds['LSTM'] = pd.Series(net.predict(Xall, verbose=0).ravel(), index=X.index)
        print('LSTM test RMSE', f"{ru.rmse(y[te], preds['LSTM'][te]):.4f}")
    except Exception as e:
        print('LSTM skipped:', repr(e))


LSTM test RMSE 1.1137


In [16]:
# --- Metrics, predictions and placeholder values ------------------------------------------------
P = pd.DataFrame(preds)
P['y_true'] = y
P['split'] = np.where(cau.index.year <= TRAIN_END_YEAR, 'train',
                      np.where(cau.index.year == VAL_YEAR, 'validation', 'test'))
P['target_observed'] = ok
P.to_csv(REV + 'original_protocol_predictions.csv', index_label='date')
rows = []
for sname, msk in (('Train', tr), ('Validation', va), ('Test', te)):
    for mname in preds:
        p = P.loc[msk, mname]
        good = p.notna()
        rows.append(dict(model=mname, split=sname, **ru.metrics(P.loc[msk, 'y_true'][good], p[good])))
T = pd.DataFrame(rows)
T.to_csv(REV + 'tables/T1_original_protocol_metrics.csv', index=False)
display(T[T.split == 'Test'].round(4))
test_rows = T[T.split == 'Test'].set_index('model')
pers = test_rows.loc['Persistence', 'RMSE']
for mname, code in ru.MODEL_CODES.items():
    if mname in test_rows.index:
        for met, f in (('RMSE', '.4f'), ('MAE', '.4f'), ('R2', '.4f'), ('MaxAE', '.4f')):
            RES.put(f'T1_{code}_{met.upper()}', test_rows.loc[mname, met], f)
        RES.put(f'T1_{code}_SKILL', 100 * (1 - test_rows.loc[mname, 'RMSE'] / pers), '.1f')
val_rows = T[T.split == 'Validation'].set_index('model')
for mname, code in ru.MODEL_CODES.items():
    if mname in val_rows.index:
        RES.put(f'VAL_{code}_RMSE', val_rows.loc[mname, 'RMSE'], '.4f')
trees = [m for m in ('Random Forest', 'LightGBM', 'CatBoost', 'XGBoost (400 trials)') if m in test_rows.index]
RES.put('T1_TREE_SPREAD', test_rows.loc[trees, 'RMSE'].max() - test_rows.loc[trees, 'RMSE'].min(), '.3f')
RES.save()


,model,split,RMSE,MAE,R2,MaxAE,n
28,Persistence,Test,1.1294,0.8338,0.2527,4.4000,364
29,Seasonal-naive,Test,1.4605,1.1312,-0.2498,5.3571,364
30,7-day moving average,Test,1.2906,0.9507,0.0242,5.7714,364
31,LassoCV,Test,1.1077,0.8590,0.2812,4.8855,364
32,RidgeCV,Test,1.1472,0.8882,0.2289,4.9584,364
33,ElasticNetCV,Test,1.0760,0.8366,0.3216,4.6194,364
34,SVR,Test,1.0675,0.8220,0.3324,4.9127,364
35,Random Forest,Test,0.9741,0.7488,0.4441,4.5551,364
36,LightGBM,Test,0.9659,0.7444,0.4534,4.4609,364
37,CatBoost,Test,0.9706,0.7474,0.4481,4.5811,364


99 values written to /content/drive/MyDrive/windforecast_rev1/revision/results_07.json
